# Standalone edition
This notebook is self-contained. It embeds the small reusable utility layer directly, so it can be uploaded and opened as a single `.ipynb` file without the companion `src/` package. Install PyTorch if your notebook environment does not already provide it.

# 02 — Weight Space Intelligence Engine
### A reusable, inspectable, Transformer-aware Python engine

**Primary demonstration architecture:** a small decoder-only PyTorch Transformer with 2 layers, width 128, 4-head causal self-attention, SwiGLU, RMSNorm, pre-norm residual connections, and tied token/output embeddings. This notebook is standalone and embeds its implementation; PyTorch is the only required non-standard dependency.

The engine uses read-only inspection by default and applies interventions only to deep-copied candidate models. Architecture-specific transformations are explicitly scoped and verified with functional probes.


In [1]:
import copy
import math
import json
import hashlib
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(11)

In [2]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        scale = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x * scale * self.weight

class CausalSelfAttention(nn.Module):
    def __init__(self, dim=128, heads=4):
        super().__init__()
        assert dim % heads == 0
        self.dim, self.heads, self.head_dim = dim, heads, dim // heads
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.o_proj = nn.Linear(dim, dim, bias=False)
    def forward(self, x):
        b, t, d = x.shape
        def split_heads(z):
            return z.view(b, t, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = map(split_heads, (self.q_proj(x), self.k_proj(x), self.v_proj(x)))
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        causal = torch.triu(torch.ones(t, t, device=x.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(causal, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=-1)
        context = weights @ v
        context = context.transpose(1, 2).contiguous().view(b, t, d)
        return self.o_proj(context)

class SwiGLU(nn.Module):
    def __init__(self, dim=128, hidden_dim=256):
        super().__init__()
        self.gate_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.up_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, dim, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class TransformerBlock(nn.Module):
    def __init__(self, dim=128, heads=4, mlp_dim=256):
        super().__init__()
        self.attn_norm = RMSNorm(dim)
        self.attn = CausalSelfAttention(dim, heads)
        self.ffn_norm = RMSNorm(dim)
        self.ffn = SwiGLU(dim, mlp_dim)
    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        x = x + self.ffn(self.ffn_norm(x))
        return x

class MiniDecoderTransformer(nn.Module):
    """Small decoder-only LM for architecture-aware experiments; not pretrained."""
    def __init__(self, vocab_size=32, max_seq_len=16, dim=128, heads=4,
                 layers=2, mlp_dim=256, tie_embeddings=True):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, max_seq_len=max_seq_len,
                          dim=dim, heads=heads, layers=layers, mlp_dim=mlp_dim,
                          tie_embeddings=tie_embeddings)
        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.position_embedding = nn.Embedding(max_seq_len, dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(dim, heads, mlp_dim) for _ in range(layers)
        ])
        self.final_norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        if tie_embeddings:
            self.lm_head.weight = self.token_embedding.weight
    def forward(self, input_ids):
        b, t = input_ids.shape
        if t > self.config['max_seq_len']:
            raise ValueError('Sequence exceeds configured max_seq_len')
        positions = torch.arange(t, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None, :, :]
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.final_norm(x))

def permute_attention_heads_(model, layer_index, permutation):
    """Function-preserving head permutation for this explicit MHA implementation."""
    attn = model.blocks[layer_index].attn
    p = torch.as_tensor(permutation, dtype=torch.long)
    if sorted(p.tolist()) != list(range(attn.heads)):
        raise ValueError('permutation must contain each attention head exactly once')
    idx = torch.cat([torch.arange(int(h)*attn.head_dim, (int(h)+1)*attn.head_dim) for h in p]).to(attn.q_proj.weight.device)
    with torch.no_grad():
        for proj in (attn.q_proj, attn.k_proj, attn.v_proj):
            proj.weight.copy_(proj.weight[idx, :])
        attn.o_proj.weight.copy_(attn.o_proj.weight[:, idx])
    return model

def permute_swiglu_units_(model, layer_index, permutation):
    """Function-preserving permutation of the paired SwiGLU intermediate units."""
    ffn = model.blocks[layer_index].ffn
    p = torch.as_tensor(permutation, dtype=torch.long)
    if sorted(p.tolist()) != list(range(ffn.gate_proj.out_features)):
        raise ValueError('permutation must contain every SwiGLU intermediate unit')
    with torch.no_grad():
        ffn.gate_proj.weight.copy_(ffn.gate_proj.weight[p])
        ffn.up_proj.weight.copy_(ffn.up_proj.weight[p])
        ffn.down_proj.weight.copy_(ffn.down_proj.weight[:, p])
    return model

@torch.no_grad()
def ablate_attention_head(model, layer_index, head_index):
    """Ablate one head's contribution by zeroing its input slice to O projection."""
    candidate = copy.deepcopy(model)
    attn = candidate.blocks[layer_index].attn
    if not 0 <= head_index < attn.heads:
        raise IndexError('head_index out of range')
    start, end = head_index * attn.head_dim, (head_index + 1) * attn.head_dim
    attn.o_proj.weight[:, start:end].zero_()
    return candidate

@torch.no_grad()
def ablate_swiglu_unit(model, layer_index, unit_index):
    candidate = copy.deepcopy(model)
    ffn = candidate.blocks[layer_index].ffn
    if not 0 <= unit_index < ffn.gate_proj.out_features:
        raise IndexError('unit_index out of range')
    ffn.gate_proj.weight[unit_index].zero_()
    ffn.up_proj.weight[unit_index].zero_()
    ffn.down_proj.weight[:, unit_index].zero_()
    return candidate


def state_dict_sha256(model: nn.Module) -> str:
    digest = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        digest.update(name.encode("utf-8"))
        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

def next_token_loss(model, input_ids, target_ids):
    logits = model(input_ids)
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1))

model = MiniDecoderTransformer().to(DEVICE).eval()
print('Model hash:', state_dict_sha256(model))
print('Configuration:', model.config)
print('Tied embedding storage:', model.lm_head.weight.data_ptr() == model.token_embedding.weight.data_ptr())


Model hash: 01557e0375b799f256e856bf6563781698d3b7b2f6e9e10259472d91e43b11ca
Configuration: {'vocab_size': 32, 'max_seq_len': 16, 'dim': 128, 'heads': 4, 'layers': 2, 'mlp_dim': 256, 'tie_embeddings': True}
Tied embedding storage: True


## 1. Inspector: architecture and tensor inventory

Statistics are descriptive, not direct evidence that a tensor is useful or harmful. The architecture summary also records the model configuration and detects shared parameter storage, which matters for tied embeddings and safe transformations.


In [3]:
class WeightSpaceInspector:
    def __init__(self, model: nn.Module):
        self.model = model
    def architecture(self) -> Dict[str, Any]:
        shared = {}
        seen = {}
        for name, parameter in self.model.named_parameters(remove_duplicate=False):
            key = (parameter.untyped_storage().data_ptr(), parameter.storage_offset(), tuple(parameter.shape))
            if key in seen:
                shared.setdefault(seen[key], []).append(name)
            else:
                seen[key] = name
        return {
            'class': self.model.__class__.__name__,
            'config': getattr(self.model, 'config', None),
            'modules': [{'name': n, 'class': m.__class__.__name__} for n, m in self.model.named_modules()],
            'parameter_count': sum(p.numel() for p in self.model.parameters()),
            'trainable_parameter_count': sum(p.numel() for p in self.model.parameters() if p.requires_grad),
            'shared_parameter_aliases': shared,
        }
    def tensor_inventory(self):
        rows = []
        for name, tensor in self.model.state_dict().items():
            t = tensor.detach().float().cpu()
            rows.append({'name': name, 'shape': list(tensor.shape), 'dtype': str(tensor.dtype),
                         'numel': tensor.numel(), 'mean': t.mean().item() if t.numel() else 0.0,
                         'std': t.std(unbiased=False).item() if t.numel() else 0.0,
                         'min': t.min().item() if t.numel() else 0.0,
                         'max': t.max().item() if t.numel() else 0.0})
        return rows

inspector = WeightSpaceInspector(model)
print(json.dumps(inspector.architecture(), indent=2))
print('Tensor inventory preview:', json.dumps(inspector.tensor_inventory()[:8], indent=2))

{
  "class": "MiniDecoderTransformer",
  "config": {
    "vocab_size": 32,
    "max_seq_len": 16,
    "dim": 128,
    "heads": 4,
    "layers": 2,
    "mlp_dim": 256,
    "tie_embeddings": true
  },
  "modules": [
    {
      "name": "",
      "class": "MiniDecoderTransformer"
    },
    {
      "name": "token_embedding",
      "class": "Embedding"
    },
    {
      "name": "position_embedding",
      "class": "Embedding"
    },
    {
      "name": "blocks",
      "class": "ModuleList"
    },
    {
      "name": "blocks.0",
      "class": "TransformerBlock"
    },
    {
      "name": "blocks.0.attn_norm",
      "class": "RMSNorm"
    },
    {
      "name": "blocks.0.attn",
      "class": "CausalSelfAttention"
    },
    {
      "name": "blocks.0.attn.q_proj",
      "class": "Linear"
    },
    {
      "name": "blocks.0.attn.k_proj",
      "class": "Linear"
    },
    {
      "name": "blocks.0.attn.v_proj",
      "class": "Linear"
    },
    {
      "name": "blocks.0.attn.o_proj",
    

## 2. Functional comparison

A comparison needs representative token sequences. Agreement on a small probe set is evidence only for those probes, not proof of global equivalence.


In [4]:
@torch.no_grad()
def compare_outputs(model_a, model_b, probe_x):
    old_a, old_b = model_a.training, model_b.training
    try:
        model_a.eval(); model_b.eval()
        out_a, out_b = model_a(probe_x), model_b(probe_x)
        if out_a.shape != out_b.shape:
            raise ValueError('Output shape mismatch')
        return {'max_abs_error': (out_a-out_b).abs().max().item(),
                'mean_abs_error': (out_a-out_b).abs().mean().item(),
                'token_prediction_agreement': (out_a.argmax(-1) == out_b.argmax(-1)).float().mean().item()}
    finally:
        model_a.train(old_a); model_b.train(old_b)

probe_x = torch.randint(0, model.config['vocab_size'], (16, 12), device=DEVICE)
comparison = compare_outputs(model, copy.deepcopy(model), probe_x)
comparison

{'max_abs_error': 0.0,
 'mean_abs_error': 0.0,
 'token_prediction_agreement': 1.0}

## 3. Gradient sensitivity

Mean absolute gradient is a local, objective- and dataset-dependent diagnostic. Near-zero gradients do not prove irrelevance. Here the objective is next-token prediction over synthetic token sequences.


In [5]:
def gradient_sensitivity(model, input_ids, target_ids):
    was_training = model.training
    model.zero_grad(set_to_none=True)
    try:
        model.train()
        logits = model(input_ids)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1))
        loss.backward()
        report = {}
        for name, parameter in model.named_parameters():
            if parameter.grad is None:
                report[name] = {'mean_abs_grad': None, 'max_abs_grad': None}
            else:
                grad = parameter.grad.detach().abs()
                report[name] = {'mean_abs_grad': grad.mean().item(), 'max_abs_grad': grad.max().item()}
        return loss.item(), report
    finally:
        model.zero_grad(set_to_none=True)
        model.train(was_training)

x = torch.randint(0, model.config['vocab_size'], (12, 12), device=DEVICE)
y = torch.randint(0, model.config['vocab_size'], (12, 12), device=DEVICE)
loss_value, sensitivity = gradient_sensitivity(model, x, y)
print('Next-token probe loss:', loss_value)
print(json.dumps({k:v for k,v in sensitivity.items() if 'blocks.0.attn' in k}, indent=2))

Next-token probe loss: 83.69979858398438
{
  "blocks.0.attn_norm.weight": {
    "mean_abs_grad": 0.018299221992492676,
    "max_abs_grad": 0.06837578862905502
  },
  "blocks.0.attn.q_proj.weight": {
    "mean_abs_grad": 0.0049506318755447865,
    "max_abs_grad": 0.036083679646253586
  },
  "blocks.0.attn.k_proj.weight": {
    "mean_abs_grad": 0.005200645420700312,
    "max_abs_grad": 0.04029799625277519
  },
  "blocks.0.attn.v_proj.weight": {
    "mean_abs_grad": 0.025280006229877472,
    "max_abs_grad": 0.19548611342906952
  },
  "blocks.0.attn.o_proj.weight": {
    "mean_abs_grad": 0.02506684698164463,
    "max_abs_grad": 0.2122647911310196
  }
}


## 4. Counterfactual intervention: ablate one attention head

This operation is architecture-aware for the explicit `MiniDecoderTransformer`: it zeros the selected head's output-channel slice entering the O projection, on a deep-copied candidate. This is an intervention, not a function-preserving transformation.


In [6]:
def evaluate_loss(model, input_ids, target_ids):
    model.eval()
    with torch.no_grad():
        logits = model(input_ids)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1)).item()

baseline_loss = evaluate_loss(model, x, y)
ablation_rows = []
for head in range(model.blocks[0].attn.heads):
    candidate = ablate_attention_head(model, layer_index=0, head_index=head)
    altered_loss = evaluate_loss(candidate, x, y)
    ablation_rows.append({'layer': 0, 'attention_head': head,
                          'baseline_loss': baseline_loss, 'ablated_loss': altered_loss,
                          'delta_loss': altered_loss-baseline_loss})
sorted(ablation_rows, key=lambda r: r['delta_loss'])

[{'layer': 0,
  'attention_head': 3,
  'baseline_loss': 83.69979858398438,
  'ablated_loss': 83.74591064453125,
  'delta_loss': 0.046112060546875},
 {'layer': 0,
  'attention_head': 0,
  'baseline_loss': 83.69979858398438,
  'ablated_loss': 83.76478576660156,
  'delta_loss': 0.0649871826171875},
 {'layer': 0,
  'attention_head': 2,
  'baseline_loss': 83.69979858398438,
  'ablated_loss': 83.89557647705078,
  'delta_loss': 0.19577789306640625},
 {'layer': 0,
  'attention_head': 1,
  'baseline_loss': 83.69979858398438,
  'ablated_loss': 83.9063491821289,
  'delta_loss': 0.20655059814453125}]

## 5. Symmetry-preserving Transformer permutations

The adapter supports two explicit transformations: attention-head permutation (Q/K/V output blocks plus O input blocks) and SwiGLU intermediate-unit permutation (paired gate/up rows plus down-projection columns). It intentionally does not claim compatibility with arbitrary Transformer implementations, GQA/MQA, fused QKV layouts, rotary variants, or third-party checkpoint conventions.


In [7]:
permuted_model = copy.deepcopy(model)
permute_attention_heads_(permuted_model, layer_index=0, permutation=[2, 0, 3, 1])
symmetry_check = compare_outputs(model, permuted_model, probe_x)
print('Attention-head permutation:', symmetry_check)
assert symmetry_check['max_abs_error'] < 1e-4

swiglu_model = copy.deepcopy(model)
permute_swiglu_units_(swiglu_model, 0, torch.randperm(model.blocks[0].ffn.gate_proj.out_features))
swiglu_check = compare_outputs(model, swiglu_model, probe_x)
print('SwiGLU permutation:', swiglu_check)
assert swiglu_check['max_abs_error'] < 1e-4

Attention-head permutation: {'max_abs_error': 3.814697265625e-05, 'mean_abs_error': 1.5222276488202624e-06, 'token_prediction_agreement': 1.0}
SwiGLU permutation: {'max_abs_error': 2.288818359375e-05, 'mean_abs_error': 1.3227257795733749e-06, 'token_prediction_agreement': 1.0}


## 6. Auditable report export

The report stores measurements and model hashes. It does not claim a component is globally irrelevant and does not modify the source checkpoint.

In [8]:
def build_report(model, inspector, comparison, sensitivity, ablations):
    return {
        "engine_version": "0.1.0",
        "model_sha256": state_dict_sha256(model),
        "architecture": inspector.architecture(),
        "tensor_inventory": inspector.tensor_inventory(),
        "functional_comparison": comparison,
        "gradient_sensitivity": sensitivity,
        "counterfactual_ablation": ablations,
        "limitations": [
            "Results are conditional on the supplied probe data and objective.",
            "Low gradient or low variance does not establish parameter irrelevance.",
            "Architecture-specific transformations require explicit compatibility checks.",
        ],
    }

report = build_report(model, inspector, symmetry_check, sensitivity, ablation_rows)
Path("artifacts").mkdir(exist_ok=True)
with open("artifacts/weight_space_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("Wrote artifacts/weight_space_report.json")

Wrote artifacts/weight_space_report.json


## 7. Optional NFN research integration

The [AllanYangZhou/nfn repository](https://github.com/AllanYangZhou/nfn) is a research implementation worth studying for permutation-equivariant weight-space processing. Treat it as an optional backend, not a universal checkpoint reader.

Before integrating:
1. Pin a commit and record its license/dependencies.
2. Run its own tests.
3. Inspect supported `network_spec` structures and tensor ordering.
4. Start with supported MLP/CNN examples.
5. Do not feed arbitrary Transformer state dictionaries into its helpers without an architecture-specific adapter.

Useful reference: [NFN test_layers.py](https://github.com/AllanYangZhou/nfn/blob/main/tests/test_layers.py).

## Engine safety contract

- Never overwrite the source model during experiments.
- Save candidate artifacts separately and record hashes before and after edits.
- Record the exact architecture adapter and transformation implementation version.
- Verify symmetry-preserving operations with output-level probes and numerical tolerances.
- Treat ablation as a scoped experiment, not proof of global irrelevance.
- Fail closed for unsupported architectures or parameter layouts.
- Keep evaluation dataset/version, runtime versions, and approval evidence in the report.


## 8. Product architecture: Model Intelligence & Assurance Engine

The engine should support several applications through one shared analysis core.

### Shared engine services

1. **Artifact intake:** PyTorch checkpoints, Safetensors, Hugging Face model metadata, hashes, provenance.
2. **Architecture adapters:** explicit tensor maps for supported MLP, CNN, and Transformer families.
3. **Weight-space representation:** tensor statistics, structured features, embeddings, optional NFN backend.
4. **Behavioral probing:** fixed, versioned input suites; optional hidden-state probes where supported.
5. **Counterfactual analysis:** ablations, controlled perturbations, sensitivity and group interventions.
6. **Security analysis:** integrity drift, suspicious structural patterns, provenance gaps, anomaly triage.
7. **Evidence reporting:** model identity, hashes, data versions, methods, findings, confidence and limitations.

### Adapter contract

Do not infer arbitrary tensor meaning from tensor names alone. Each architecture adapter must declare:
- model family and supported versions
- parameter names and shapes
- layer / head / neuron mappings
- tied-weight relationships
- valid symmetry transformations
- intervention and reconstruction rules
- known unsupported features

This is the foundation for safely extending from small MLP demonstrations to real Transformer checkpoints.

## 9. Product demo A — Model Artifact Security Assessment

**Customer problem:** a team receives a model checkpoint from a vendor, open-source repository, internal training run, or fine-tuning pipeline. They need evidence about what they received before deployment.

### Demo workflow
1. Load the artifact and metadata.
2. Calculate cryptographic hashes and record provenance.
3. Inspect architecture, tensor shapes, parameter counts and numerical statistics.
4. Compare against a trusted baseline where available.
5. Run structural anomaly heuristics and optional learned weight-space diagnostics.
6. Run a declared behavioral probe suite.
7. Produce a finding report with evidence and uncertainty.

### Example report fields
- Artifact identity and hash
- Architecture adapter and coverage
- Unexpected tensor / shape differences
- Parameter distribution shifts
- Probe-suite results
- Suspected anomaly indicators
- False-positive caveats
- Recommended next tests

**Security boundary:** anomaly detection is triage, not proof of a backdoor. No automatic blocking based solely on a statistical outlier.

## 10. Product demo B — Model Change Impact & Optimization

**Customer problem:** a model has been fine-tuned, merged, quantized, compressed, or edited. The team wants to know whether the change matters and whether it is safe to promote.

### Demo workflow
1. Compare source and candidate hashes and metadata.
2. Align models only when the architecture and symmetry assumptions permit it.
3. Compare raw parameter differences and functional outputs separately.
4. Run layer / head / neuron interventions on supported architectures.
5. Measure loss, task metrics, latency, memory, and regression-suite behavior.
6. Generate a promotion recommendation with supporting evidence.

### Product outputs
- Change-impact report
- Candidate redundancy hypotheses
- Capability-specific sensitivity map
- Performance / memory comparison
- Regression results
- Rollback reference

The engine must not equate low gradient, low variance, or low impact on one dataset with global irrelevance.

## 11. Product demo C — Model Discovery & Selection

**Customer problem:** an organization has many models and checkpoints but lacks a consistent way to identify which are appropriate for a task.

Potential research integrations:
- WeightCLIP-style joint model / dataset embeddings.
- Functional probe fingerprints.
- Architecture and provenance metadata.
- Task-specific benchmark profiles.
- Similarity search over previously assessed artifacts.

A model should not be selected solely because its weight embedding is close to a dataset embedding. The result is a candidate ranking that must be verified with task-specific evaluation.

## 12. Product demo D — Model Optimization Workbench

Potential integrations:
- NFN / equivariant weight-space modules for supported architectures.
- LoRA and conventional fine-tuning baselines.
- Compression and structured factorization.
- Candidate weight edits with immutable originals.
- Generative weight methods such as DeepWeightFlow as a research track.

The workbench compares methods against the same baseline and evaluation contract. It should report quality, latency, memory, artifact size, cost estimate, and safety regressions.

### Engine API direction

The eventual Python package should expose stable operations such as:
- `inspect_artifact(...)`
- `fingerprint_model(...)`
- `compare_models(...)`
- `run_probe_suite(...)`
- `analyze_sensitivity(...)`
- `assess_security_posture(...)`
- `evaluate_candidate_edit(...)`
- `export_evidence_report(...)`

These are interface targets; implement each only when its supported scope and tests are defined.